# PointDiT in motion: Kaggle evaluation

CPU only (Accelerator: None), Internet on. Attach the "PointDiT TUM RGB-D Subset" dataset and the output of the runner notebook version that wrote the predictions (it holds `predictions/<run name>/run.json`). Add the Kaggle secret `GITHUB_TOKEN` while the repo is private. No GPU, DINOv3, weights or `HF_TOKEN` are needed.

With `RUN_EVALUATION = False` (default) the notebook does not re-evaluate: also attach the output of the evaluation notebook version that wrote `eval/<run name>/per_frame.csv`, and it reuses that CSV. With `RUN_EVALUATION = True` it runs `evaluate.py` (including `rel_d_near/mid/far` and `focal_ratio`) into `/kaggle/working/eval/<run name>/`; if an older `per_frame.csv` of the same run is attached, `compare_eval_csv.py` then checks that every shared column is reproduced within 1e-6 and stops the notebook otherwise. The preview step writes RGB | predicted depth | GT depth | relative error panels and `error_breakdown.csv` to `/kaggle/working/previews/<run name>/`.

In [ ]:
PROJECT_REPO = "https://github.com/Sainava/Pointdit-In-Motion.git"
PROJECT_BRANCH = "feat/analysis-compute"
# None: use the single run.json found under /kaggle/input; set a path if several are attached.
PREDICTIONS_DIR = None
# False: reuse the single per_frame.csv found under /kaggle/input instead of running evaluate.py (~1.5 h).
# True: run evaluate.py, then compare with the per_frame.csv attached under /kaggle/input, if any.
RUN_EVALUATION = False
# Each worker needs ~3.5 GB of RAM; torch already uses every core for one frame.
WORKERS = 1

PROJECT_DIR = "/kaggle/working/Pointdit-In-Motion"
FORK_DIR = f"{PROJECT_DIR}/third_party/pointdit"
SEQUENCES_CONFIG = f"{PROJECT_DIR}/configs/sequences.yaml"

import base64
import os
import subprocess
import sys

from kaggle_secrets import UserSecretsClient


def github_token() -> str | None:
    try:
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


# Only needed while the repo is private. Git reads the token from the environment, so it
# never appears in a command, in cell output, or in the clone's .git/config.
token = github_token()
if token:
    basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    os.environ["GIT_CONFIG_COUNT"] = "1"
    os.environ["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    os.environ["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {basic_auth}"
    del basic_auth
print("GitHub auth:", "token" if token else "none (public repo)")
del token

if os.path.isdir(f"{PROJECT_DIR}/.git"):
    !git -C {PROJECT_DIR} fetch -q origin {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} checkout -q {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} pull -q --ff-only origin {PROJECT_BRANCH}
else:
    !rm -rf {PROJECT_DIR}
    !git clone --recurse-submodules --branch {PROJECT_BRANCH} {PROJECT_REPO} {PROJECT_DIR}
if not os.path.isdir(f"{PROJECT_DIR}/.git"):
    raise RuntimeError(f"Cloning {PROJECT_REPO} failed; see the git output above.")
!git -C {PROJECT_DIR} submodule update --init --recursive
%cd {PROJECT_DIR}
!echo "project $(git -C {PROJECT_DIR} rev-parse HEAD)"
!echo "fork    $(git -C {FORK_DIR} rev-parse HEAD)"


def run(*cmd: object) -> None:
    """Run a project script, streaming its output, and fail the cell if it fails."""
    proc = subprocess.Popen([sys.executable, *map(str, cmd)], cwd=PROJECT_DIR, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd[0]} exited with code {proc.returncode}; see the output above")

In [ ]:
!pip install -q "numpy>=1.26.0" "Pillow>=10.0.0" "matplotlib>=3.9.0" "PyYAML>=6.0" "torch>=2.4"

import torch

print("torch", torch.__version__, "| CPUs", os.cpu_count(), "| CUDA", torch.cuda.is_available())

In [ ]:
import csv
import json
from pathlib import Path

sys.path.insert(0, f"{PROJECT_DIR}/src")
from pim.data.tum import find_sequences

sequences = sorted(find_sequences("/kaggle/input"), key=lambda s: s.name)
for sequence in sequences:
    print(sequence)
data_roots = {s.parent for s in sequences}
if len(data_roots) != 1:
    raise RuntimeError(f"sequences are spread over several folders: {sorted(data_roots)}")
DATA_ROOT = data_roots.pop()

if PREDICTIONS_DIR is None:
    runs = sorted(Path("/kaggle/input").rglob("run.json"))
    if len(runs) != 1:
        raise RuntimeError(f"expected one run.json under /kaggle/input, found {[str(r) for r in runs]}; "
                           "set PREDICTIONS_DIR in the first cell")
    RUN_DIR = runs[0].parent
else:
    RUN_DIR = Path(PREDICTIONS_DIR)
run_record = json.loads((RUN_DIR / "run.json").read_text())
RUN_NAME = run_record["name"]
EVAL_DIR = Path("/kaggle/working/eval") / RUN_NAME
print(f"\npredictions {RUN_DIR} (run {RUN_NAME!r}, {run_record.get('start_time')} -> {run_record.get('end_time')})")
for name, seq in run_record["sequences"].items():
    print(f"  {name}: window {seq['window']}, {len(list((RUN_DIR / name).glob('*.npz')))} npz files")
print(f"results -> {EVAL_DIR}")

In [ ]:
def attached_per_frame_csvs() -> list[Path]:
    """per_frame.csv files of attached inputs only; never one this session wrote to /kaggle/working."""
    working = Path("/kaggle/working").resolve()
    found = sorted(Path("/kaggle/input").rglob("per_frame.csv"))
    written_here = [p for p in found if p.resolve().is_relative_to(working)]
    if written_here:
        raise RuntimeError(f"{[str(p) for p in written_here]} under /kaggle/input resolve into /kaggle/working")
    return found


def check_evaluates_this_run(csv_path: Path) -> None:
    eval_json = csv_path.parent / "eval.json"
    if not eval_json.is_file():
        raise RuntimeError(f"{eval_json} not found next to {csv_path}; cannot tell which run it evaluated")
    evaluated_run = json.loads(eval_json.read_text())["predictions"]["run"]
    if evaluated_run != run_record:
        raise RuntimeError(f"{csv_path} evaluates run {evaluated_run.get('name')!r} started "
                           f"{evaluated_run.get('start_time')}, but the attached predictions are run {RUN_NAME!r} "
                           f"started {run_record.get('start_time')}")


# Look for the attached CSV before evaluating, so the new CSV cannot be among the candidates.
attached = attached_per_frame_csvs()
if RUN_EVALUATION:
    if len(attached) > 1:
        raise RuntimeError(f"found several per_frame.csv under /kaggle/input: {[str(p) for p in attached]}; "
                           "attach at most one older evaluation to compare against")
    for old_csv in attached:
        check_evaluates_this_run(old_csv)
    run(f"{PROJECT_DIR}/scripts/evaluate.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--out", EVAL_DIR, "--workers", WORKERS)
    CSV_PATH = EVAL_DIR / "per_frame.csv"
    if attached:
        print(f"\ncomparing {CSV_PATH} with {attached[0]}")
        run(f"{PROJECT_DIR}/scripts/compare_eval_csv.py", CSV_PATH, attached[0])
    else:
        print("\nno older per_frame.csv attached under /kaggle/input; skipping the reproduction check")
else:
    if len(attached) != 1:
        raise RuntimeError(f"RUN_EVALUATION is False, so exactly one per_frame.csv must be attached under /kaggle/input; "
                           f"found {[str(p) for p in attached]}. Attach the output of the evaluation notebook version "
                           "that evaluated this run, or set RUN_EVALUATION = True.")
    CSV_PATH = attached[0]
    check_evaluates_this_run(CSV_PATH)
print(f"per-frame results {CSV_PATH}")


In [ ]:
with open(CSV_PATH, newline="") as f:
    rows = list(csv.DictReader(f))
evaluated = list(dict.fromkeys(r["sequence"] for r in rows))

for name in evaluated:
    window = run_record["sequences"][name]["window"]
    middle = window["start"] + window["count"] // 2
    run(f"{PROJECT_DIR}/scripts/check_frame.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--csv", CSV_PATH, "--sequence", name, "--frame", middle)
    print()

In [ ]:
import numpy as np

WORST = 5
ROOM = "rgbd_dataset_freiburg1_room"
ROOM_FRAMES = range(366, 381)
PREVIEW_DIR = Path("/kaggle/working/previews") / RUN_NAME
BREAKDOWN_CSV = PREVIEW_DIR / "error_breakdown.csv"

if ROOM not in evaluated:
    raise RuntimeError(f"{CSV_PATH} has no {ROOM} rows; evaluated sequences are {evaluated}")


def preview_frames(name: str) -> dict[int, str]:
    """Frame index -> why it is previewed."""
    scored = sorted((r for r in rows if r["sequence"] == name and r["rel_d"]), key=lambda r: float(r["rel_d"]))
    if not scored:
        raise RuntimeError(f"{name}: no frame with a rel_d in {CSV_PATH}")
    median = float(np.median([float(r["rel_d"]) for r in scored]))
    closest = min(scored, key=lambda r: abs(float(r["rel_d"]) - median))
    reasons: dict[int, list[str]] = {}
    for rank, r in enumerate(reversed(scored[-WORST:]), start=1):
        reasons.setdefault(int(r["frame_index"]), []).append(f"worst{rank}")
    reasons.setdefault(int(closest["frame_index"]), []).append("median")
    if name == ROOM:
        for index in ROOM_FRAMES:
            reasons.setdefault(index, []).append("room366-380")
    return {index: "+".join(why) for index, why in sorted(reasons.items())}


# The script merges into an existing error_breakdown.csv; start clean so it holds exactly this frame list.
BREAKDOWN_CSV.unlink(missing_ok=True)
reasons = {}
for name in evaluated:
    frames = preview_frames(name)
    reasons.update({(name, index): why for index, why in frames.items()})
    print(name, frames)
    run(f"{PROJECT_DIR}/scripts/preview_predictions.py", "--predictions", RUN_DIR, "--data-root", DATA_ROOT,
        "--sequences-config", SEQUENCES_CONFIG, "--sequence", name, "--frames", *frames,
        "--csv", CSV_PATH, "--out", PREVIEW_DIR)
    print()


In [ ]:
with open(BREAKDOWN_CSV, newline="") as f:
    breakdown = list(csv.DictReader(f))


def number(value: str, digits: int) -> str:
    return f"{float(value):.{digits}f}" if value else "n/a"


print("band_px / band_err: share of valid pixels / of total relative error within 5 px of a GT depth edge or hole.")
print("band_err >> band_px: error sits on edges; band_err ~ band_px with high rel_out: error spread over regions.")
for name in evaluated:
    print(f"\n{name}")
    print(f"{'frame':>6} {'rel_d':>7} {'band_px':>8} {'band_err':>8} {'ratio':>6} {'rel_band':>8} {'rel_out':>8} "
          f"{'offset_ms':>9}  why")
    for r in (r for r in breakdown if r["sequence"] == name):
        ratio = (f"{float(r['band_error_fraction']) / float(r['band_pixel_fraction']):.2f}"
                 if r["band_error_fraction"] and float(r["band_pixel_fraction"]) > 0 else "n/a")
        print(f"{r['frame_index']:>6} {number(r['rel_d'], 2):>7} {number(r['band_pixel_fraction'], 3):>8} "
              f"{number(r['band_error_fraction'], 3):>8} {ratio:>6} {number(r['rel_d_band'], 2):>8} "
              f"{number(r['rel_d_outside'], 2):>8} {number(r['depth_offset_ms'], 1):>9}  "
              f"{reasons[name, int(r['frame_index'])]}")

print("\nto download:")
for directory in (EVAL_DIR, PREVIEW_DIR):
    for path in sorted(directory.rglob("*")) if directory.is_dir() else []:
        if path.is_file():
            print(f"  {path} ({path.stat().st_size / 1e3:.0f} kB)")
